# RAG Agent Using LangChain

## 1. Install packages

In [ ]:
%pip install -q \
langchain \
langchain-community \
langchain-google-genai \
langchain-huggingface \
langchain-chroma \
langchain-tavily \
sentence-transformers \
pypdf

## 2. Load API keys from Colab Secrets

In [ ]:
from google.colab import userdata
import os

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is missing. Add it in Colab Secrets and enable notebook access.")

if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is missing. Add it in Colab Secrets and enable notebook access.")

# The integrations read credentials from environment variables.
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

print("API keys loaded successfully (values are not displayed).")

API keys loaded successfully (values are not displayed).


## 3. Upload PDF

In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
if not pdf_files:
    raise ValueError("Please upload at least one PDF file.")

PDF_PATH = pdf_files[0]
print("Selected PDF:", PDF_PATH)

## 4. Load and split PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print("Pages loaded:", len(documents))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

all_splits = text_splitter.split_documents(documents)
print("Total chunks created:", len(all_splits))

if not all_splits:
    raise ValueError("No text was extracted from the PDF. Check that the PDF contains selectable text.")

/tmp/ipykernel_2540/3620768509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages loaded: 15
Total chunks created: 52


## 5. Create embeddings and Chroma vector store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Create a fresh in-memory collection for this Colab run.
# This avoids accidentally mixing chunks from an earlier uploaded PDF.
vector_store = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    collection_name="rag_agent_collection"
)

print("Vector database created successfully.")
print("Number of chunks indexed:", len(all_splits))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created successfully.
Number of chunks indexed: 52


## 6. Test PDF retrieval

In [ ]:
def test_retrieval(query: str, k: int = 3):
    results = vector_store.similarity_search(query, k=k)
    for i, doc in enumerate(results, start=1):
        page = doc.metadata.get("page", "unknown")
        print(f"\\n--- Result {i} | PDF page {page} ---")
        print(doc.page_content[:1000])
    return results

test_retrieval("What is the attention mechanism in the Transformer architecture?")

\n--- Result 1 | PDF page 4 ---
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
\n--- Result 2 | PDF page 2 ---
Figure 1: The Transfor

[Document(id='ef9f35fe-844c-4b24-895f-9a0f9662fed6', metadata={'start_index': 838, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'trapped': '/False', 'creator': 'LaTeX with hyperref', 'author': '', 'page_label': '5', 'page': 4, 'keywords': '', 'producer': 'pdfTeX-1.40.25', 'subject': '', 'moddate': '2024-04-10T21:11:43+00:00', 'total_pages': 15, 'title': '', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': 'attention_is_all_you_need (1).pdf'}, page_content='The Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,\nand the memory keys and values come from the output of the encoder. This allows every\nposition in the decoder to attend over all positions in the input sequence. This mimics the\ntypical encoder-decoder attention mechanisms in sequence-to-sequence models such as\n[38, 2, 9].\n• The encoder contains self-att

## 7. Create PDF retrieval tool

In [ ]:
from langchain.tools import tool

@tool
def retrieve_from_pdf(query: str) -> str:
    '''Search the uploaded PDF and return the most relevant passages with page metadata.
    Use this tool for information that may be contained in the uploaded document.
    '''
    retrieved_docs = vector_store.similarity_search(query, k=4)

    if not retrieved_docs:
        return "No relevant passages were found in the uploaded PDF."

    output = []
    for i, doc in enumerate(retrieved_docs, start=1):
        page = doc.metadata.get("page", "unknown")
        # PyPDFLoader page numbers are zero-based; display human-friendly page numbers.
        display_page = page + 1 if isinstance(page, int) else page
        output.append(
            f"Source {i} | PDF page {display_page}\n{doc.page_content}"
        )

    return "\n\n".join(output)

print("PDF retrieval tool created:", retrieve_from_pdf.name)

PDF retrieval tool created: retrieve_from_pdf


## 8. Create Tavily web-search tool

In [ ]:
from langchain_tavily import TavilySearch

web_search_tool = TavilySearch(
    max_results=5,
    search_depth="advanced"
)

print("Web search tool created:", web_search_tool.name)

Web search tool created: tavily_search


## 9. Initialize Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("Gemini model initialized successfully.")

Gemini model initialized successfully.


## 10. Define agent instructions

In [ ]:
system_prompt = """
You are a careful research assistant for a PDF-based DocuChat application.

You have two tools:
1. retrieve_from_pdf: searches the PDF uploaded by the user.
2. tavily_search: searches the public web for current or external information.

Tool-use strategy:
- For questions about the uploaded paper or its specific contents, use retrieve_from_pdf.
- For recent developments, current facts, or topics not covered by the PDF, use web search.
- For comparison questions involving both the paper and newer developments, use BOTH tools.
- If the PDF does not contain the requested information, say so clearly. Do not pretend that web information came from the PDF.
- Never invent facts, sources, quotations, or page numbers.
- Base the answer on retrieved evidence. Explain uncertainty or conflicting evidence.
- Give a clear, well-structured answer that directly addresses every part of the user's question.
- When possible, mention the PDF page number for claims drawn from the uploaded document.
"""

print("Agent instructions ready.")

Agent instructions ready.


## 11. Create RAG agent

In [ ]:
from langchain.agents import create_agent

tools = [retrieve_from_pdf, web_search_tool]

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt
)

print("RAG Agent created successfully.")

RAG Agent created successfully.


## 12. Define question-answering function

In [ ]:

def ask_rag_agent(user_query: str, show_tool_activity: bool = True):

    response = agent.invoke({
        "messages": [{"role": "user", "content": user_query}]
    })

    messages = response.get("messages", [])

    if show_tool_activity:
        print("TOOL ACTIVITY")
        for message in messages:
            for call in getattr(message, "tool_calls", []):
                print("-", call.get("name"), "called")

    print("\nFINAL ANSWER\n")

    for message in reversed(messages):
        if message.__class__.__name__ == "AIMessage" and message.content:

            content = message.content

            if isinstance(content, list):
                for block in content:
                    if isinstance(block, dict) and block.get("type") == "text":
                        print(block.get("text", ""))
            else:
                print(content)

            return response

    print("No final answer found.")
    return response


## 13. Test the RAG agent

In [ ]:
user_query = (
    "Compare the attention mechanism from the paper with recent improvements "
    "like Flash Attention, and tell me which approach would be better for my college project."
)

result = ask_rag_agent(user_query)

TOOL ACTIVITY
- retrieve_from_pdf called
- tavily_search called

FINAL ANSWER

The attention mechanism described in the paper, "Attention Is All You Need," is a fundamental concept in Transformer models. It functions by mapping a query and a set of key-value pairs to an output, where the output is a weighted sum of the values. This mechanism allows the model to focus on different parts of the input sequence when processing each element, effectively capturing long-range dependencies within the data (Source 1, page 3). The paper illustrates how attention heads can follow these dependencies and even perform tasks like anaphora resolution (Source 2, 3, 4, pages 13-15).

However, the standard attention mechanism, as originally conceived, has practical limitations:
*   **Quadratic Memory Usage:** Its memory complexity is O(N²) with respect to the sequence length (N), meaning memory requirements grow rapidly with longer inputs. This can lead to out-of-memory errors on GPUs for very long seque

## 14. Try additional questions

In [ ]:
# Example A: Mostly about the uploaded paper
ask_rag_agent("Explain scaled dot-product attention as described in the uploaded paper.")

# Example B: About a more recent topic
# ask_rag_agent("What are the latest developments in efficient Transformer inference?")

# Example C: Requires both sources
# ask_rag_agent("What does the uploaded paper say about multi-head attention, and how is it different from modern efficient attention methods?")

TOOL ACTIVITY
- retrieve_from_pdf called

FINAL ANSWER

Scaled Dot-Product Attention, as described in the paper, is an attention mechanism that takes queries (Q), keys (K) of dimension `dk`, and values (V) of dimension `dv` as input (page 4).

The process involves the following steps:
1.  **Dot Product Calculation:** The dot products of the query with all keys are computed.
2.  **Scaling:** Each dot product is then divided by the square root of `dk` (`√dk`). This scaling factor is crucial because for large values of `dk`, the dot products can become large in magnitude, pushing the softmax function into regions with extremely small gradients, which can impede effective learning (page 4).
3.  **Softmax Application:** A softmax function is applied to these scaled dot products to obtain the weights that will be assigned to the values.
4.  **Weighted Sum:** The output is then computed as a weighted sum of the values, where the weights are determined by the compatibility function of the quer

{'messages': [HumanMessage(content='Explain scaled dot-product attention as described in the uploaded paper.', additional_kwargs={}, response_metadata={}, id='963cb9b0-c0b5-499b-a2bb-0710da6e3a19'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'retrieve_from_pdf', 'arguments': '{"query": "Explain scaled dot-product attention"}'}, '__gemini_function_call_thought_signatures__': {'4f1eb782-6157-4551-afba-3fdfbaf2a1cf': 'CrcCAWkUfRP5BjP+0VF2XbHtQn0kLKJbgc3Hf3Cti564fwZeFnn1r0fhzTG+fsz93CVsgxLVA2y4GDq0XxqaakMFkiZW21ESuft97ukaKgq6nCFIBw8ikKDl5B+R6eyBrDIU/Fr3wtEb5p6pOLKBQe8xkgYiwAitYFNqFPPBDAlWOyDcQVizMCyLLePKVa6GxwwALvgiPzuNELYkxcg2m/6GTU3j7tS6PYdfeAii6cP6Lm1+rPjUmO2pp2Vio59StzsbHLLkbWLJoCVIJaUfnSvWG43o80PylsJfMIVMgFYQ+9BFLya2qHDlRZLmmra6X/6b8Vrdmvp0vUvhbYC6C5v2iblOk/fcR5hhIkqZyj54L2FAuBf+bZ2CsCOU3ZAlGitEcWzxdllBNp7nBGefmbY+sRelCE6m/UQ='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_gena